# Topology optimisation solver: five acceptance checks

Before a topology-optimisation code may design a chair, it has to get five simpler things right: the deflection of a clamped beam (in 2D and 3D), the rigid-body modes of its 3D element, the gradient it uses to move material, the answer of its fast multigrid solver, and the four published MBB-beam designs of Andreassen et al. (2011). This notebook runs all five with the solver of the phyein-topopt repository and compares every number with beam theory, automatic differentiation, a direct solver and the paper.

This project has no page on <https://phyeinusul.com> yet.

Code: <https://github.com/Numa1997/phyein-topopt> at commit `ef2d27b`.

**Computed results; this page is not interactive. Press Copy & Edit to run it yourself.**

Contents: 1 The physics · 2 The machine · 3 The code · 4 Parameters · 5 The run · 6 Results · 7 Verification · 8 Files

## 1. The physics

*Source of this section: the derivation in `skills/topology-optimisation/SKILL.md`, sections 2.1 to 2.11, and the docstrings of the four scripts in `benchmarks/`, all in the repository at the commit named above. The text follows them; the beam-theory steps in 1.4 are written out here in full. Phyein Usul has no lecture note on topology optimisation yet, so this repository text is the note.*

### 1.1 What the code must get right, and why these five tests

Clamp a steel ruler to a table edge and press its free end. It bends, and the end sinks by an amount you can predict with a pencil: that is beam theory. A topology-optimisation code does the same calculation millions of times, on a block cut into small cubes, and after each one it moves a little material from where it does little to where it carries load. If the calculation is wrong, or the direction in which it moves material is wrong, the shape it draws means nothing, however good it looks.

So the code is tested in the order it is built. (A) Does the finite-element model of a clamped beam sag as much as beam theory says, in 2D and in 3D? (B) Is the 3D building block, one cube, free to move rigidly and stiff against every deformation? (C) Is the gradient that steers the material the true derivative of the objective? (D) Does the fast iterative solver return the same displacements as a slow exact one? (E) Does the whole optimiser reproduce the four MBB-beam designs published with the 88-line code?

### 1.2 Elastic equilibrium

A body occupies the region $\Omega$. Some faces are held ($\Gamma_D$, displacement $\mathbf u = 0$), some are pushed by a traction $\mathbf t$ (force per area, N/m², on $\Gamma_N$), and a body force $\mathbf b$ (N/m³) acts inside. Each material point $\mathbf x$ moves to $\mathbf x + \mathbf u(\mathbf x)$.

Cut out any piece $V$. At rest the surface forces $\boldsymbol\sigma\mathbf n$ and the body force balance: $\oint_{\partial V}\boldsymbol\sigma\mathbf n\,dA + \int_V \mathbf b\,dV = 0$, with $\boldsymbol\sigma$ the stress tensor and $\mathbf n$ the outward normal. The divergence theorem turns the surface integral into a volume integral, and because $V$ is arbitrary the integrand vanishes:

$$\nabla\cdot\boldsymbol\sigma + \mathbf b = 0 \ \text{in}\ \Omega,\qquad \boldsymbol\sigma\mathbf n = \mathbf t \ \text{on}\ \Gamma_N,\qquad \mathbf u = 0 \ \text{on}\ \Gamma_D. \tag{1}$$

The balance of moments on the same piece makes $\boldsymbol\sigma$ symmetric.

For displacement gradients much smaller than 1, stretching and shearing are measured by the symmetric part of $\nabla\mathbf u$; the antisymmetric part is a rigid rotation and stores no energy:

$$\boldsymbol\varepsilon = \tfrac12\left(\nabla\mathbf u + \nabla\mathbf u^{\mathsf T}\right). \tag{2}$$

Squeeze a rubber eraser and it widens sideways; shear it and it skews. For an isotropic linear material two constants describe both, Young's modulus $E$ (Pa) and Poisson's ratio $\nu$:

$$\boldsymbol\sigma = \lambda\,\mathrm{tr}(\boldsymbol\varepsilon)\,\mathbf I + 2\mu\,\boldsymbol\varepsilon,\qquad \lambda = \frac{E\nu}{(1+\nu)(1-2\nu)},\qquad \mu = G = \frac{E}{2(1+\nu)}. \tag{3}$$

In Voigt notation, $\boldsymbol\sigma = \mathbf D\boldsymbol\varepsilon$ with a $6\times6$ matrix $\mathbf D$ in 3D; in 2D plane stress ($\sigma_{zz} = 0$, a thin plate) $\mathbf D$ is $3\times3$.

Multiply (1) by any test displacement $\mathbf v$ that vanishes on $\Gamma_D$, integrate over $\Omega$, and integrate the stress term by parts. The boundary term becomes $\int_{\Gamma_N}\mathbf t\cdot\mathbf v\,dA$, and with the symmetry of $\boldsymbol\sigma$:

$$\int_\Omega \boldsymbol\sigma(\mathbf u):\boldsymbol\varepsilon(\mathbf v)\,dV = \int_\Omega \mathbf b\cdot\mathbf v\,dV + \int_{\Gamma_N}\mathbf t\cdot\mathbf v\,dA\qquad\text{for all admissible }\mathbf v. \tag{4}$$

This is virtual work: the internal work of the stresses in a virtual deformation equals the external work of the loads. It is also the condition that the potential energy $\Pi(\mathbf u) = \tfrac12\int\boldsymbol\varepsilon:\mathbf D:\boldsymbol\varepsilon\,dV - \int\mathbf b\cdot\mathbf u\,dV - \int\mathbf t\cdot\mathbf u\,dA$ is stationary, and a minimum because $\mathbf D$ is positive definite.

### 1.3 Finite elements

Fill the domain with squares (2D, four-node "Q4" elements) or cubes (3D, eight-node "hex8") of side $a$. Inside one element, in natural coordinates $\xi,\eta,\zeta\in[-1,1]$, the displacement is interpolated from the corner displacements with the shape functions $N_A = \tfrac18(1+\xi\xi_A)(1+\eta\eta_A)(1+\zeta\zeta_A)$ in 3D (the 2D ones drop $\zeta$ and the factor becomes $\tfrac14$). Then $\boldsymbol\varepsilon = \mathbf B\mathbf u_e$, where $\mathbf u_e$ holds the element's corner displacements (8 numbers in 2D, 24 in 3D) and $\mathbf B$ the derivatives $\partial N_A/\partial x$. Insert $\mathbf u\approx\sum N_A\mathbf u_A$ and $\mathbf v\approx\sum N_A\mathbf v_A$ into (4); because it holds for every choice of the $\mathbf v_A$, it becomes one linear equation per degree of freedom:

$$\mathbf K\mathbf u = \mathbf f,\qquad \mathbf K = \sum_e \mathbf A_e^{\mathsf T}\mathbf k_e\mathbf A_e,\qquad \mathbf k_e = \int_{\Omega_e}\mathbf B^{\mathsf T}\mathbf D\,\mathbf B\,dV, \tag{5}$$

where $\mathbf A_e$ picks the element's degrees of freedom out of the global vector and $\mathbf f$ collects the loads. For a cube, $\partial/\partial x = (2/a)\,\partial/\partial\xi$ and $dV = (a/2)^3 d\xi\,d\eta\,d\zeta$, so

$$\mathbf k_e = E\,a\,\hat{\mathbf k}\quad(\text{3D}),\qquad \mathbf k_e = E\,t\,\hat{\mathbf k}\quad(\text{2D, thickness } t), \tag{6}$$

with $\hat{\mathbf k}$ built once for a unit element with $E = 1$. In 2D the element size drops out entirely. The 2D code integrates $\hat{\mathbf k}$ with $2\times2$ Gauss points (the closed form of the 88-line paper, its eq. 12). The 3D code splits $\mathbf D$ into a deviatoric part, integrated with $2\times2\times2$ points, and a volumetric part, integrated with one point at the centre ("selective reduced integration", Hughes 1980, *Int. J. Numer. Meth. Eng.* 15:1413). The reason: full integration of the trilinear cube cannot change shape at constant volume, so it is far too stiff for a material like PA12 with $\nu = 0.43$.

### 1.4 Test A: the clamped beam against beam theory

**Picture.** A beam of length $L$, clamped at $x = 0$, with a load $F$ pulling its free end down. Look at a cut at position $x$: everything to the right of the cut is held in equilibrium by a shear force $V(x)$ and a bending moment $M(x)$ carried across the cut. The end sinks for two reasons. The beam curves (bending), and each slice skews slightly (shear).

**Bending (Euler–Bernoulli).** Balance the piece between $x$ and the tip: $V(x) = F$ and $M(x) = F\,(L - x)$. Plane sections stay plane and normal to the axis, so the curvature is $w''(x) = M(x)/(E I)$, with $w$ the downward deflection and $I$ the second moment of area of the cross-section. Integrate twice with the clamp conditions $w(0) = 0$, $w'(0) = 0$:

$$w'(x) = \frac{F}{EI}\Big(Lx - \frac{x^2}{2}\Big),\qquad w(x) = \frac{F}{EI}\Big(\frac{Lx^2}{2} - \frac{x^3}{6}\Big),\qquad \delta_{EB} = w(L) = \frac{F L^3}{3EI}. \tag{7}$$

**Shear (Timoshenko).** The shear force $V = F$ skews every slice by an average angle $\gamma = V/(\kappa G A)$, where $A$ is the area of the section and $\kappa$ a shear coefficient that accounts for the shear stress being uneven over the section. The slices add up along the length, $w_s(L) = \int_0^L \gamma\,dx$:

$$\delta_{T} = \frac{F L^3}{3EI} + \frac{F L}{\kappa G A}. \tag{8}$$

The two sections tested, with their $\kappa$:

* 2D (`benchmarks/cantilever2d.py`): a plate of height $h$ and thickness $t$ in plane stress, $I = t h^3/12$, $A = t h$, $\kappa = 5/6$.
* 3D (`benchmarks/check3d.py`): a square section $h\times h$, $I = h^4/12$, $A = h^2$, and Cowper's coefficient for a rectangle, $\kappa = 10(1+\nu)/(12+11\nu)$ (Cowper 1966, *J. Appl. Mech.* 33:335).

**Check.** The ratio of the shear term to the bending term in (8) is $3EI/(\kappa G A L^2) = (E/G)\,h^2/(4\kappa L^2)$ for a rectangle. It vanishes like $(h/L)^2$ for a slender beam, as it should. With $L/h = 10$, $E/G = 2(1+\nu) = 2.6$ and $\kappa = 5/6$ it is $2.6\times0.01/3.33 = 0.78\,\%$: shear adds less than 1 % here, which is why the test compares with (8) and not with (7) alone.

**What to expect.** The test uses $E = F = t = 1$ and elements of unit side, so $h$ equals the number of elements $n_h$ across the section and $L = 10\,n_h$. In 2D, $\delta_{EB} = F L^3/(3EI) = 4000$ for every $n_h$. Bilinear elements are too stiff in bending (they cannot bend without also shearing, "shear locking"), so the finite-element deflection approaches (8) from below as $n_h$ grows. The fully clamped root also stops the Poisson contraction there, which beam theory ignores. Pass: within 1 % of (8) at the finest mesh.

### 1.5 Test B: the rigid-body modes of one cube

**Picture.** Push a loose dice across the table, or spin it: it moves, but nothing in it is strained, so it stores no energy. A free cube can do this in 6 independent ways (3 translations, 3 rotations), and no others. In the language of (5) this means $\hat{\mathbf k}$ must be symmetric, have no negative eigenvalue, and have exactly 6 zero eigenvalues. A seventh zero would be a deformation that costs nothing (an "hourglass mode", which makes the model floppy); fewer than 6 would mean a coding error. Selective reduced integration is exactly the kind of change that can create hourglass modes, which is why this test exists.

### 1.6 Test C: the design problem and its gradient

Give every element a density $\rho_e\in[0,1]$ and let its Young's modulus be (SIMP, Bendsøe 1989, *Struct. Optim.* 1:193)

$$E_e = E_{\min} + \rho_e^{\,p}\,(E_0 - E_{\min}),\qquad p \approx 3,\quad E_{\min} = 10^{-9}E_0. \tag{9}$$

The objective is the compliance $C = \mathbf f^{\mathsf T}\mathbf u$, the work done by the loads. Press on a structure: the stiffer it is, the less the load sinks and the less work it does. The design problem is

$$\min_{\boldsymbol\rho}\ C(\boldsymbol\rho) = \mathbf f^{\mathsf T}\mathbf u(\boldsymbol\rho)\quad\text{subject to}\quad \mathbf K(\boldsymbol\rho)\,\mathbf u = \mathbf f,\quad \sum_e\rho_e v_e \le V^*\sum_e v_e,\quad 0\le\rho_e\le1. \tag{10}$$

Why $p > 1$: with $p = 1$ a half-dense element gives half the stiffness for half the material, so grey is a fair buy; with $p = 3$ it gives one eighth of the stiffness, so the optimiser drives elements to 0 or 1.

The optimiser needs $\partial C/\partial\rho_e$ for every element, at the cost of one solve. Differentiate $\mathbf K\mathbf u = \mathbf f$ ($\mathbf f$ does not depend on $\boldsymbol\rho$): $(\partial\mathbf K/\partial\rho_e)\mathbf u + \mathbf K\,\partial\mathbf u/\partial\rho_e = 0$. Then, with $\mathbf K$ symmetric and $\mathbf K^{-1}\mathbf f = \mathbf u$,

$$\frac{\partial C}{\partial\rho_e} = \mathbf f^{\mathsf T}\frac{\partial\mathbf u}{\partial\rho_e} = -\mathbf u^{\mathsf T}\frac{\partial\mathbf K}{\partial\rho_e}\mathbf u = -\,p\,\rho_e^{\,p-1}(E_0 - E_{\min})\;\mathbf u_e^{\mathsf T}\hat{\mathbf k}\,\mathbf u_e\ \le 0. \tag{11}$$

In words: the gradient for each element is minus its own strain energy, up to the SIMP factor, and never positive, because adding material can only stiffen. This is the adjoint method; for compliance the adjoint vector equals $\mathbf u$ itself, so no second solve is needed.

**The test.** On a small 3D grid ($4\times3\times2$ cubes, random densities between 0.2 and 1, $E_{\min} = 10^{-3}$, the face $x = 0$ clamped, two point loads at the free end) the code computes (11), and JAX computes the derivative of $C(\boldsymbol\rho) = \mathbf f^{\mathsf T}\mathbf K(\boldsymbol\rho)^{-1}\mathbf f$ by automatic differentiation in 64-bit arithmetic, without knowing (11). Pass: the largest difference, relative to the largest gradient, below $10^{-10}$.

### 1.7 Test D: the fast solver against the exact one

Solving (5) is the cost of everything. The code solves it without ever building $\mathbf K$: every cube shares one $\hat{\mathbf k}$, so $\mathbf K\mathbf u$ is one gather, one small dense product and one scatter. Conjugate gradients converge fast on this when they are preconditioned by geometric multigrid (Amir, Aage & Lazarov 2014, *Struct. Multidisc. Optim.* 49:815): the error is smoothed on the fine grid, and its smooth part is removed on coarser grids, where it looks rough again. Picture smoothing a crumpled sheet: first iron the small creases, then step back and pull out the big waves.

**The test.** The chair's load cases, at 8 voxels across the half width, with a uniform density 0.05, are solved by this multigrid solver to a tolerance of $10^{-10}$ and by the direct sparse solver PARDISO, which factorises $\mathbf K$ exactly. Pass: relative difference of the displacement fields below $10^{-6}$.

### 1.8 Test E: the MBB beam against the 88-line paper

**Picture.** A long bridge deck on two supports, loaded at mid-span from above. By symmetry only the left half is modelled: the left edge may slide up and down but not sideways (the mirror plane), the bottom-right corner rests on a roller, and a unit load pushes down at the top-left corner. The task is to place half the material ($V^* = 0.5$) so the half-beam is as stiff as possible.

Two diseases need cures (Sigmund & Petersson 1998, *Struct. Optim.* 16:68). Low-order elements make a chessboard of solid and void look artificially stiff, so the optimiser draws one; and the continuum problem has no length scale, so finer meshes give ever finer members. Both are cured by a filter of radius $r_{\min}$, which makes members at least about $2r_{\min}$ wide. The paper uses two:

* sensitivity filter ($f_t = 1$, Sigmund 1997): $\widetilde{\dfrac{\partial C}{\partial\rho_e}} = \dfrac{\sum_i H_{ei}\,\rho_i\,\partial C/\partial\rho_i}{\max(\gamma,\rho_e)\sum_i H_{ei}}$, with $\gamma = 10^{-3}$;
* density filter ($f_t = 2$, Bruns & Tortorelli 2001; Bourdin 2001): $\tilde\rho_e = \dfrac{\sum_i H_{ei}\rho_i}{\sum_i H_{ei}}$, gradients by the chain rule;

both with the weights $H_{ei} = \max(0,\ r_{\min} - |\mathbf x_e - \mathbf x_i|)$. The density update is the optimality-criteria step: at an optimum every grey element satisfies $\partial C/\partial\rho_e + \Lambda\,\partial V/\partial\rho_e = 0$, with $\Lambda$ the price of material. The step

$$\rho_e \leftarrow \rho_e\,B_e^{1/2},\qquad B_e = \frac{-\partial C/\partial\rho_e}{\Lambda\,\partial V/\partial\rho_e}, \tag{12}$$

clipped to a move limit $m = 0.2$ and to $[0,1]$, with $\Lambda$ found by bisection so that the volume is exactly $V^*$. Picture a market: elements whose strain energy is worth more than the price of material get more material, the others less. The loop stops when no design variable changes by more than 0.01.

**The test.** The paper's Fig. 3 gives the final compliance for $60\times20$ and $150\times50$ elements with each filter, at $p = 3$, $\nu = 0.3$ and $r_{\min} = 0.04\,n_{elx}$ (the value its numbers use, not the 1.5 of its example call): 216.81 and 233.71 ($60\times20$, $f_t = 1, 2$), 219.52 and 235.73 ($150\times50$) (Andreassen, Clausen, Schevenels, Lazarov & Sigmund 2011, *Struct. Multidisc. Optim.* 43:1, doi:10.1007/s00158-010-0594-7). The Python port in `topopt/top2d.py` follows the MATLAB code line by line. Pass: relative error below $10^{-4}$ for all four.

## 2. The machine

What this run had to work with. The same lines go into `env.json` and `timing.json`.

In [ ]:
import os, sys, json, time, platform, subprocess, datetime, hashlib
from pathlib import Path

T0 = time.time()
STARTED_UTC = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
WORK = Path(os.environ.get("NB_WORK", "/kaggle/working"))   # Kaggle keeps this folder as the notebook's output
SRC = Path(os.environ.get("NB_SRC", str(Path.home() / "src" / "topopt-benchmarks")))   # code, not kept as output
WORK.mkdir(parents=True, exist_ok=True)
NEED_GPU = False
TIMES = {}          # wall time of every stage, seconds
CHECKS = {}         # verification results, filled in section 7


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None


def log(msg):
    line = f"{datetime.datetime.now(datetime.timezone.utc):%Y-%m-%dT%H:%M:%SZ}  {time.time() - T0:8.1f} s  {msg}"
    print(line, flush=True)
    with open(WORK / "progress.log", "a") as f:
        f.write(line + "\n")


class stage:
    """with stage("name"): ... times a block, logs its start and end, and records it in TIMES."""
    def __init__(self, name): self.name = name
    def __enter__(self):
        self.t = time.time(); log(f"start {self.name}"); return self
    def __exit__(self, *exc):
        TIMES[self.name] = round(time.time() - self.t, 3); log(f"end   {self.name}  ({TIMES[self.name]:.1f} s)")


def meminfo_gb(key):
    for line in open("/proc/meminfo"):
        if line.startswith(key + ":"):
            return round(int(line.split()[1]) / 2**20, 2)


def cgroup_limit_gb():
    for p in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            v = open(p).read().strip()
            return None if v == "max" or int(v) > 2**60 else round(int(v) / 2**30, 2)
        except (OSError, ValueError):
            pass


cpu_model = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), None)
ENV = dict(
    started_utc=STARTED_UTC,
    on_kaggle=bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")),
    kaggle_run_type=os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
    os=platform.platform(),
    python=sys.version.split()[0],
    cpu_model=cpu_model,
    nproc=os.cpu_count(),
    cpus_usable=len(os.sched_getaffinity(0)),
    ram_total_gb=meminfo_gb("MemTotal"),
    ram_available_gb=meminfo_gb("MemAvailable"),
    ram_cgroup_limit_gb=cgroup_limit_gb(),
    disk_free_work_gb=round(os.statvfs(WORK).f_bavail * os.statvfs(WORK).f_frsize / 2**30, 1),
    gpu=sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader"),
)
for k, v in ENV.items():
    print(f"{k:22s} {v}")
if NEED_GPU and not ENV["gpu"]:
    raise RuntimeError("A GPU was requested but nvidia-smi finds none: stopping before any work is done.")
log("environment checked")

## 3. The code

The solver is not copied into this notebook: it is fetched from GitHub at one fixed commit, so this page always runs the code it describes. Only packages the Kaggle image lacks are installed, at the versions pinned in the repository's `requirements.txt`; every version actually used is written to `env.json`.

In [ ]:
import importlib.metadata as md

REPO = "https://github.com/Numa1997/phyein-topopt"
COMMIT = "ef2d27b0f620bfc3d0c40ca691ac2bac2f79ea46"
REQUIRES = ["numpy", "scipy", "matplotlib", "pyamg", "pypardiso", "mkl", "jax"]   # the packages this notebook imports, as named in requirements.txt

with stage("clone"):
    if not (SRC / ".git").exists():
        SRC.mkdir(parents=True, exist_ok=True)
        for cmd in (["git", "init", "-q"], ["git", "remote", "add", "origin", REPO],
                    ["git", "fetch", "-q", "--depth", "1", "origin", COMMIT], ["git", "checkout", "-q", "FETCH_HEAD"]):
            subprocess.run(cmd, cwd=SRC, check=True)
    head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=SRC, capture_output=True, text=True, check=True).stdout.strip()
    assert head == COMMIT, f"checked out {head}, expected {COMMIT}"
    log(f"code at {REPO} commit {head}")

# Pinned versions from the repository at that commit ("name==version" or "name[extra]==version").
PINS = {}
for line in (SRC / "requirements.txt").read_text().splitlines():
    line = line.split("#", 1)[0].strip()
    if "==" in line:
        spec_, ver = line.split("==", 1)
        PINS[spec_.split("[", 1)[0].strip().lower()] = (line, ver.strip())


def installed(dist):
    try:
        return md.version(dist)
    except md.PackageNotFoundError:
        return None


with stage("install"):
    missing = [PINS[p][0] for p in REQUIRES if installed(p) is None]
    if missing:
        log("installing (missing from this image): " + " ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input", *missing], check=True)
    else:
        log("nothing to install: the image has every package")
    import importlib
    importlib.invalidate_caches()
    fetched = {m.split("==", 1)[0].split("[", 1)[0].strip().lower() for m in missing}
    ENV["packages"] = {p: dict(used=installed(p), pinned=PINS[p][1],
                               source="installed here" if p in fetched else "already in the image")
                       for p in REQUIRES}
    for p, v in ENV["packages"].items():
        flag = "" if v["used"] == v["pinned"] else "   (differs from the pin; see section 7)"
        print(f"{p:12s} used {v['used']:14s} pinned {v['pinned']:14s} {v['source']}{flag}")

sys.path[:0] = [str(SRC), str(SRC / "benchmarks"), str(SRC / "examples")]
os.environ.setdefault("MPLBACKEND", "Agg")

## 4. Parameters

Every input of the run, as a named variable. Change one and run the notebook again; the values used are written to `params.json`.

In [ ]:
# Material and load. The tests are dimensionless: deflections scale as F / (E t), compliances as F^2 / E.
E0 = 1.0             # Young's modulus of solid material, E_0 in (9)
E_MIN = 1e-9         # Young's modulus of void, E_min in (9)
NU = 0.3             # Poisson's ratio, nu in (3)
F = 1.0              # tip load of the cantilevers, F in (7)-(8)
T = 1.0              # plate thickness of the 2D cantilever, t in (6)

# Test A: cantilever against beam theory, (7)-(8)
L_OVER_H = 10                    # length / height of the beam
N_H_2D = [2, 4, 8, 16, 32]       # elements across the height, 2D
N_H_3D = [2, 4, 8, 16]           # elements across the square section, 3D
CANTILEVER_TOL = 0.01            # pass: |FE / Timoshenko - 1| below this at the finest mesh

# Test B: rigid-body modes of the hex8 element (uses NU)
ZERO_MODES_EXPECTED = 6

# Test C: adjoint gradient (11) against JAX automatic differentiation
GRAD_SEED = 0                    # random densities in [0.2, 1] from this seed
GRAD_PENAL = 3.0                 # p in (9)
GRAD_EMIN = 1e-3                 # E_min in (9) for this test (a stiffer void keeps the dense solve well conditioned)
GRAD_TOL = 1e-10

# Test D: multigrid-preconditioned CG against the direct solver PARDISO
MG_N = 8                         # chair grid: voxels across the half width
MG_RHO = 0.05                    # uniform density
MG_PENAL = 3.0
MG_TOL = 1e-6                    # pass: relative difference of the displacement fields

# Test E: MBB beam against Andreassen et al. (2011), Fig. 3
MBB_CASES = [(60, 20, 1), (60, 20, 2), (150, 50, 1), (150, 50, 2)]   # (nelx, nely, filter type)
VOLFRAC = 0.5                    # V* in (10)
PENAL = 3.0                      # p in (9)
RMIN_FACTOR = 0.04               # r_min = RMIN_FACTOR * nelx, as in the paper's Fig. 3
MOVE = 0.2                       # move limit m of the update (12)
OC_TOL = 0.01                    # stop when no design variable changes by more than this
MAXIT = 2000
MBB_PAPER = {"60x20 ft1": 216.81, "60x20 ft2": 233.71, "150x50 ft1": 219.52, "150x50 ft2": 235.73}
MBB_TOL = 1e-4                   # relative; the paper prints five significant digits

PRECISION = "float64 throughout (NumPy, SciPy, PARDISO); JAX with jax_enable_x64 for the gradient check"

PARAMS = {k: globals()[k] for k in [
    "E0", "E_MIN", "NU", "F", "T", "L_OVER_H", "N_H_2D", "N_H_3D", "CANTILEVER_TOL", "ZERO_MODES_EXPECTED",
    "GRAD_SEED", "GRAD_PENAL", "GRAD_EMIN", "GRAD_TOL", "MG_N", "MG_RHO", "MG_PENAL", "MG_TOL",
    "MBB_CASES", "VOLFRAC", "PENAL", "RMIN_FACTOR", "MOVE", "OC_TOL", "MAXIT", "MBB_PAPER", "MBB_TOL", "PRECISION"]}


# Write every parameter above into params.json, together with where the code came from.
json.dump(dict(PARAMS, _source=dict(repo=REPO, commit=COMMIT), _project="topopt-benchmarks"),
          open(WORK / "params.json", "w"), indent=1, default=str)
for k, v in PARAMS.items():
    print(f"{k:24s} {v}")
log("parameters written to params.json")

## 5. The run

Each stage appends a line to `progress.log` with the elapsed time.

In [ ]:
import contextlib, io, re
import numpy as np
from topopt.top2d import top2d
from topopt.top3d import hex8_stiffness
import cantilever2d, check3d, check_mgcg      # the repository's acceptance checks, used as a library
from examples.chair import build


def captured(fn, *a, **kw):
    """Run fn, return (its result, what it printed); the printed lines also go to this notebook."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        out = fn(*a, **kw)
    print(buf.getvalue(), end="")
    return out, buf.getvalue()


R = {}

with stage("A1 cantilever 2D"):
    R["cantilever2d"] = []
    for n in N_H_2D:
        fe, eb, tim = cantilever2d.tip_deflection(L_OVER_H, n, E=E0, nu=NU, F=F, t=T)
        R["cantilever2d"].append(dict(n_h=n, delta_fe=fe, delta_eb=eb, delta_tim=tim, rel_eb=fe / eb - 1, rel_tim=fe / tim - 1))
        log(f"2D cantilever n_h={n:3d}: FE {fe:.4f}, Euler-Bernoulli {eb:.4f}, Timoshenko {tim:.4f}, FE/Tim-1 {fe / tim - 1:+.4%}")

with stage("A2 cantilever 3D"):
    R["cantilever3d"] = []
    for n in N_H_3D:
        fe, eb, tim = check3d.cantilever(n, Lh=L_OVER_H, nu=NU, E=E0, F=F)
        R["cantilever3d"].append(dict(n_h=n, delta_fe=fe, delta_eb=eb, delta_tim=tim, rel_eb=fe / eb - 1, rel_tim=fe / tim - 1))
        log(f"3D cantilever n_h={n:3d}: FE {fe:.4f}, Euler-Bernoulli {eb:.4f}, Timoshenko {tim:.4f}, FE/Tim-1 {fe / tim - 1:+.4%}")

with stage("B element modes"):
    k0 = hex8_stiffness(NU)
    ev = np.linalg.eigvalsh(k0)
    ok_repo, _ = captured(check3d.check_element, NU)          # the repository's own verdict
    R["element3d"] = dict(sym_err=float(np.abs(k0 - k0.T).max()), min_eig=float(ev.min()), max_eig=float(ev.max()),
                          zero_modes=int(np.sum(np.abs(ev) < 1e-10 * ev.max())), eigenvalues=ev.tolist(),
                          repo_check_pass=bool(ok_repo))
    log(f"hex8 element: {R['element3d']['zero_modes']} zero modes, symmetry error {R['element3d']['sym_err']:.1e}")

with stage("C gradient"):
    ok_repo, text = captured(check3d.check_gradient, seed=GRAD_SEED, penal=GRAD_PENAL, nu=NU, Emin=GRAD_EMIN)
    rel = float(re.search(r"= ([0-9.eE+-]+) over (\d+) elements", text).group(1))
    R["gradient"] = dict(rel_diff=rel, repo_check_pass=bool(ok_repo))
    log(f"adjoint vs automatic differentiation: max relative difference {rel:.2e}")

with stage("D multigrid vs direct"):
    prob, _ = build(MG_N)
    worst, text = captured(check_mgcg.compare, MG_N, np.full(prob.grid.nel, MG_RHO), f"uniform rho={MG_RHO}",
                           nu=NU, penal=MG_PENAL, Emin=E_MIN)
    groups = [dict(group=int(g), cg_iterations=int(it), rel_diff=float(d))
              for g, it, d in re.findall(r"group (\d+): levels \d+, CG its (\d+), rel\. diff ([0-9.eE+-]+)", text)]
    R["mgcg"] = dict(worst_rel_diff=float(worst), groups=groups, nel=int(prob.grid.nel))
    log(f"multigrid vs PARDISO: worst relative difference {worst:.2e} over {len(groups)} support sets")

with stage("E MBB beam"):
    R["mbb"] = []
    for nx, ny, ft in MBB_CASES:
        t = time.time()
        r = top2d(nx, ny, VOLFRAC, PENAL, RMIN_FACTOR * nx, ft, E0=E0, Emin=E_MIN, nu=NU,
                  move=MOVE, tol=OC_TOL, maxit=MAXIT, verbose=False)
        key = f"{nx}x{ny} ft{ft}"
        ref = MBB_PAPER[key]
        np.savez_compressed(WORK / f"mbb_{nx}x{ny}_ft{ft}.npz", xPhys=r["xPhys"], c=r["c"], iters=r["iters"])
        R["mbb"].append(dict(case=key, nelx=nx, nely=ny, ft=ft, c=r["c"], paper=ref, rel_err=r["c"] / ref - 1,
                             iters=r["iters"], volume=float(r["xPhys"].mean()), seconds=round(time.time() - t, 2)))
        log(f"MBB {key}: c = {r['c']:.4f} (paper {ref:.2f}, rel. err {r['c'] / ref - 1:+.2e}), {r['iters']} iterations")

## 6. Results

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from topopt.style import BEIGE, INK, WINE, DENSITY_CMAP

json.dump(R, open(WORK / "results.json", "w"), indent=1)

# Figure 1: the four MBB designs, mirrored to show the whole beam
fig, axs = plt.subplots(2, 2, figsize=(11, 3.7), facecolor=BEIGE)
for ax, row in zip(axs.T.flat, R["mbb"]):
    x = np.load(WORK / f"mbb_{row['nelx']}x{row['nely']}_ft{row['ft']}.npz")["xPhys"]
    ax.imshow(np.hstack([x[:, ::-1], x]), cmap=DENSITY_CMAP, vmin=0, vmax=1, interpolation="nearest")
    ax.axis("off")
    ax.set_title(f"{row['nelx']}×{row['nely']}, {'sensitivity' if row['ft'] == 1 else 'density'} filter: "
                 f"C = {row['c']:.2f} (paper {row['paper']:.2f}, {100 * row['rel_err']:+.4f} %)", fontsize=9, color=INK)
fig.suptitle("Figure 1. MBB beam, V* = 0.5, p = 3, r_min = 0.04·nelx (whole beam: the computed left half and its mirror image)",
             color=INK, fontsize=10)
fig.tight_layout(); fig.savefig(WORK / "fig1_mbb_designs.png", dpi=150, facecolor=BEIGE); plt.show()

# Figure 2: convergence of the cantilever tip deflection towards Timoshenko's value (8)
fig, ax = plt.subplots(figsize=(6.4, 4.2), facecolor=BEIGE)
ax.set_facecolor(BEIGE)
for key, label, mk in (("cantilever2d", "2D, Q4 plane stress, κ = 5/6", "o"),
                       ("cantilever3d", "3D, hex8 (selective reduced integration), Cowper κ", "s")):
    n = [r["n_h"] for r in R[key]]; e = [abs(r["rel_tim"]) for r in R[key]]
    ax.loglog(n, e, marker=mk, color=WINE if key == "cantilever3d" else INK, label=label)
ax.axhline(CANTILEVER_TOL, color="#8a7f76", ls="--", lw=1, label=f"pass limit {CANTILEVER_TOL:.0%}")
ax.set_xticks(N_H_2D); ax.set_xticklabels([str(n) for n in N_H_2D]); ax.minorticks_off()
ax.set_xlabel("elements across the section, $n_h$"); ax.set_ylabel(r"$|\delta_{FE}/\delta_T - 1|$")
ax.set_title(f"Figure 2. Clamped beam, L/h = {L_OVER_H}: finite elements against beam theory", fontsize=10, color=INK)
ax.legend(fontsize=8, frameon=False); fig.tight_layout()
fig.savefig(WORK / "fig2_cantilever_convergence.png", dpi=150, facecolor=BEIGE); plt.show()

print("Test A  2D cantilever:   " + ", ".join(f"n_h={r['n_h']}: {r['rel_tim']:+.4%}" for r in R["cantilever2d"]))
print("Test A  3D cantilever:   " + ", ".join(f"n_h={r['n_h']}: {r['rel_tim']:+.4%}" for r in R["cantilever3d"]))
print(f"Test B  hex8 element:    {R['element3d']['zero_modes']} zero eigenvalues, smallest {R['element3d']['min_eig']:+.1e}, "
      f"symmetry error {R['element3d']['sym_err']:.1e}")
print(f"Test C  gradient:        max |AD - adjoint| / max |AD| = {R['gradient']['rel_diff']:.2e}")
print(f"Test D  multigrid:       worst relative difference to PARDISO {R['mgcg']['worst_rel_diff']:.2e}, "
      f"CG iterations {[g['cg_iterations'] for g in R['mgcg']['groups']]}")
for r in R["mbb"]:
    print(f"Test E  MBB {r['case']:11s} C = {r['c']:.4f}  paper {r['paper']:.2f}  rel. err {r['rel_err']:+.2e}  "
          f"{r['iters']} iterations, {r['seconds']:.1f} s")
log("results.json and figures written")

## 7. Verification

Three comparisons. **Against theory and the paper**: each check has the pass limit the repository's own acceptance checks use (MBB within $10^{-4}$ of the paper, cantilever within 1 % of Timoshenko at the finest mesh, 6 rigid-body modes, gradient within $10^{-10}$ of automatic differentiation, multigrid within $10^{-6}$ of the direct solve). **Against the numbers the repository records** in `README.md` and `HANDOVER.md`, at the precision they are printed. **Against a reference run of this same notebook** made on 12 November 2026 in a Claude Code session (4 CPU cores, Python 3.13, the pinned package versions), stored in `kaggle/projects/topopt-benchmarks/reference.json`: the largest relative difference over every deterministic number is printed. A difference above the limit is a bug to find, not a number to explain away.

In [ ]:
def check(name, value, limit, ok, note=""):
    CHECKS[name] = {"value": value, "limit": limit, "pass": bool(ok), "note": note}
    print(f"{'PASS' if ok else 'FAIL'}  {name:58s} {value!s:>24}   limit {limit}   {note}")


print("1) Against theory, automatic differentiation, the direct solver and the paper")
e = max(abs(r["rel_err"]) for r in R["mbb"])
check("MBB: largest |C / C_paper - 1| over 4 cases", f"{e:.2e}", MBB_TOL, e < MBB_TOL)
r2, r3 = R["cantilever2d"][-1], R["cantilever3d"][-1]
check(f"2D cantilever: FE / Timoshenko - 1 at n_h = {r2['n_h']}", f"{r2['rel_tim']:+.4%}", f"±{CANTILEVER_TOL:.0%}",
      abs(r2["rel_tim"]) < CANTILEVER_TOL)
check(f"3D cantilever: FE / Timoshenko - 1 at n_h = {r3['n_h']}", f"{r3['rel_tim']:+.4%}", f"±{CANTILEVER_TOL:.0%}",
      abs(r3["rel_tim"]) < CANTILEVER_TOL)
el = R["element3d"]
check("hex8 element: zero eigenvalues (rigid-body modes)", el["zero_modes"], ZERO_MODES_EXPECTED,
      el["zero_modes"] == ZERO_MODES_EXPECTED and el["sym_err"] < 1e-12 and el["min_eig"] > -1e-10 * el["max_eig"],
      f"symmetry error {el['sym_err']:.1e}, smallest eigenvalue {el['min_eig']:+.1e}")
check("gradient: max |AD - adjoint| / max |AD|", f"{R['gradient']['rel_diff']:.2e}", GRAD_TOL,
      R["gradient"]["rel_diff"] < GRAD_TOL)
check("multigrid: relative difference to PARDISO", f"{R['mgcg']['worst_rel_diff']:.2e}", MG_TOL,
      R["mgcg"]["worst_rel_diff"] < MG_TOL)

print("\n2) Against the numbers the repository records, at the precision they are printed")
RECORDED = [   # (name, value as printed, where, this run's value, how to round it)
    ("MBB largest relative error", "2e-05", "HANDOVER.md, 'Verified'", e, lambda v: f"{v:.0e}"),
    ("2D cantilever, % from Timoshenko at 32", "-0.23", "README.md, 'Verified'", 100 * R["cantilever2d"][-1]["rel_tim"],
     lambda v: f"{v:.2f}"),
    ("3D cantilever, % from Timoshenko at 16", "-0.74", "HANDOVER.md, 'Verified' (README.md says -0.93, from before "
     "selective reduced integration)", 100 * R["cantilever3d"][-1]["rel_tim"], lambda v: f"{v:.2f}"),
]
for name, printed, where, v, fmt in RECORDED:
    check(f"recorded: {name}", fmt(v), f"recorded {printed}", fmt(v) == printed, where)

print("\n3) Against a reference run of this notebook (see the text above)")
REFERENCE = {'machine': 'Intel(R) Xeon(R) Processor @ 2.10GHz, 4 CPUs, 15.71 GB, '
            'Linux-6.18.44-fc-v378-x86_64-with-glibc2.39, Python 3.13.18',
 'made': '2026-11-12, Claude Code cloud session, first local run of this notebook',
 'packages': {'jax': '0.12.0.1',
              'matplotlib': '3.11.2',
              'mkl': '2026.2.0',
              'numpy': '2.5.3',
              'pyamg': '5.4.1',
              'pypardiso': '0.4.7',
              'scipy': '1.18.1'},
 'results': {'cantilever2d': [{'delta_eb': 4000.0,
                               'delta_fe': 3581.1044700266625,
                               'delta_tim': 4031.2,
                               'n_h': 2,
                               'rel_eb': -0.10472388249333442,
                               'rel_tim': -0.11165298917774791},
                              {'delta_eb': 4000.0,
                               'delta_fe': 3900.887152749367,
                               'delta_tim': 4031.2,
                               'n_h': 4,
                               'rel_eb': -0.02477821181265827,
                               'rel_tim': -0.032326068478525705},
                              {'delta_eb': 4000.0,
                               'delta_fe': 3991.791737674381,
                               'delta_tim': 4031.2,
                               'n_h': 8,
                               'rel_eb': -0.002052065581404805,
                               'rel_tim': -0.009775814230407631},
                              {'delta_eb': 4000.0,
                               'delta_fe': 4015.746278870807,
                               'delta_tim': 4031.2,
                               'n_h': 16,
                               'rel_eb': 0.003936569717701843,
                               'rel_tim': -0.0038335287579859267},
                              {'delta_eb': 4000.0,
                               'delta_fe': 4021.9867019389585,
                               'delta_tim': 4031.2,
                               'n_h': 32,
                               'rel_eb': 0.005496675484739644,
                               'rel_tim': -0.002285497633717326}],
             'cantilever3d': [{'delta_eb': 2000.0,
                               'delta_fe': 1966.7497680736838,
                               'delta_tim': 2015.3,
                               'n_h': 2,
                               'rel_eb': -0.016625115963158077,
                               'rel_tim': -0.02409082118112249},
                              {'delta_eb': 1000.0,
                               'delta_fe': 993.2309622766135,
                               'delta_tim': 1007.65,
                               'n_h': 4,
                               'rel_eb': -0.006769037723386551,
                               'rel_tim': -0.014309569516584686},
                              {'delta_eb': 500.0,
                               'delta_fe': 499.2387986820426,
                               'delta_tim': 503.825,
                               'n_h': 8,
                               'rel_eb': -0.0015224026359147436,
                               'rel_tim': -0.009102766472400936},
                              {'delta_eb': 250.0,
                               'delta_fe': 250.0580841275425,
                               'delta_tim': 251.9125,
                               'n_h': 16,
                               'rel_eb': 0.00023233651016996149,
                               'rel_tim': -0.007361349168689579}],
             'element3d': {'eigenvalues': [-1.0440271779856317e-16,
                                           -4.216662365819688e-17,
                                           -3.983256076830525e-17,
                                           -3.043878669825408e-17,
                                           5.908185206659468e-17,
                                           8.983651703416272e-17,
                                           0.06410256410256414,
                                           0.06410256410256421,
                                           0.07122507122507107,
                                           0.07122507122507118,
                                           0.07122507122507134,
                                           0.10683760683760671,
                                           0.1068376068376068,
                                           0.10683760683760687,
                                           0.19230769230769218,
                                           0.19230769230769235,
                                           0.1923076923076924,
                                           0.2564102564102562,
                                           0.38461538461538436,
                                           0.3846153846153844,
                                           0.38461538461538464,
                                           0.3846153846153847,
                                           0.38461538461538475,
                                           1.2500000000000004],
                           'max_eig': 1.2500000000000004,
                           'min_eig': -1.0440271779856317e-16,
                           'repo_check_pass': True,
                           'sym_err': 1.3877787807814457e-17,
                           'zero_modes': 6},
             'gradient': {'rel_diff': 6.96e-15, 'repo_check_pass': True},
             'mbb': [{'c': 216.81366296246892,
                      'case': '60x20 ft1',
                      'ft': 1,
                      'iters': 106,
                      'nelx': 60,
                      'nely': 20,
                      'paper': 216.81,
                      'rel_err': 1.6894804063083413e-05,
                      'seconds': 1.36,
                      'volume': 0.4998668996255922},
                     {'c': 233.71457280775255,
                      'case': '60x20 ft2',
                      'ft': 2,
                      'iters': 144,
                      'nelx': 60,
                      'nely': 20,
                      'paper': 233.71,
                      'rel_err': 1.9566162134943e-05,
                      'seconds': 1.97,
                      'volume': 0.5000038591141646},
                     {'c': 219.51987863518283,
                      'case': '150x50 ft1',
                      'ft': 1,
                      'iters': 95,
                      'nelx': 150,
                      'nely': 50,
                      'paper': 219.52,
                      'rel_err': -5.528645097285789e-07,
                      'seconds': 18.93,
                      'volume': 0.5001285410275288},
                     {'c': 235.7336521214756,
                      'case': '150x50 ft2',
                      'ft': 2,
                      'iters': 362,
                      'nelx': 150,
                      'nely': 50,
                      'paper': 235.73,
                      'rel_err': 1.549281583002049e-05,
                      'seconds': 81.5,
                      'volume': 0.500000709620807}],
             'mgcg': {'groups': [{'cg_iterations': 12, 'group': 0, 'rel_diff': 1.38e-12},
                                 {'cg_iterations': 13, 'group': 1, 'rel_diff': 2.48e-13}],
                      'nel': 4096,
                      'worst_rel_diff': 1.382782384571289e-12}},
 'source_commit': 'ef2d27b0f620bfc3d0c40ca691ac2bac2f79ea46'}


def deterministic(res):
    """Every number that must come out the same on any machine, up to rounding: name -> value."""
    out = {}
    for key in ("cantilever2d", "cantilever3d"):
        for r in res[key]:
            for q in ("delta_fe", "delta_eb", "delta_tim"):
                out[f"{key} n_h={r['n_h']} {q}"] = r[q]
    for r in res["mbb"]:
        out[f"mbb {r['case']} C"] = r["c"]
        out[f"mbb {r['case']} iterations"] = r["iters"]
        out[f"mbb {r['case']} volume"] = r["volume"]
    ev = res["element3d"]["eigenvalues"]
    for i, v in enumerate(ev):
        if abs(v) > 1e-10 * max(ev):      # the six zero eigenvalues are rounding noise, compared by count instead
            out[f"hex8 eigenvalue {i}"] = v
    return out


if REFERENCE is None:
    print("no reference stored yet: this run is the reference")
    CHECKS_REFERENCE = None
else:
    a, b = deterministic(R), deterministic(REFERENCE["results"])
    assert a.keys() == b.keys(), "the reference holds different quantities"
    diffs = {k: abs(a[k] - b[k]) / max(abs(b[k]), 1e-300) for k in a}
    worst = max(diffs, key=diffs.get)
    print(f"reference made {REFERENCE['made']} on: {REFERENCE['machine']}")
    print(f"{len(diffs)} numbers compared; {sum(d == 0 for d in diffs.values())} identical to the last bit")
    check("reference run: largest relative difference", f"{diffs[worst]:.2e}", 1e-8, diffs[worst] < 1e-8,
          f"in '{worst}'")
    CHECKS["reference run: largest relative difference"]["all"] = diffs

json.dump(CHECKS, open(WORK / "verification.json", "w"), indent=1, default=str)
log(f"verification: {sum(c['pass'] for c in CHECKS.values())} of {len(CHECKS)} checks passed")

## 8. Files

The files this run wrote, with their SHA-256 checksums (also in `SHA256SUMS.txt`). The last line stops the notebook with an error if any verification check failed.

In [ ]:
ENV["finished_utc"] = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
TIMES["total"] = round(time.time() - T0, 3)
json.dump(ENV, open(WORK / "env.json", "w"), indent=1, default=str)
json.dump(dict(machine=dict((k, ENV[k]) for k in ("cpu_model", "cpus_usable", "ram_total_gb", "gpu", "on_kaggle")),
               precision=PRECISION, stages_s=TIMES, wall_s=TIMES["total"]),
          open(WORK / "timing.json", "w"), indent=1)
log(f"timing.json written, total {TIMES['total']:.1f} s")

files = sorted(p for p in WORK.rglob("*") if p.is_file() and p.name != "SHA256SUMS.txt")
with open(WORK / "SHA256SUMS.txt", "w") as f:
    for p in files:
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        f.write(f"{h}  {p.relative_to(WORK)}\n")
        print(f"{h[:16]}…  {p.stat().st_size:>10,d} B  {p.relative_to(WORK)}")

failed = [k for k, v in CHECKS.items() if not v["pass"]]
print(f"\n{len(CHECKS) - len(failed)} of {len(CHECKS)} verification checks passed.")
if failed:
    raise RuntimeError("verification failed: " + ", ".join(failed))